## 필독!!!

<h3> 여기 있는 코드는 절대 실행하지 마십시오. </h3>

눈으로만 보고 이해하시거나

복붙하셔서 실제 linux나 파이썬 환경에서 실행해 주시기 바랍니다.

이 파일은 jupyter 파일입니다.

여기 있는 코드는 모두 jupyter가 아닌 실제 파이썬 및 ROS2 환경에서 사용할 수 있는 코드로 작성하였습니다.

ROS2 코드를 jupyter에서 실행하는 방법이 없는 것은 아니나 별도의 방법이 따로 존재하기 때문에(`jupyter_ws` 참고)

여기 있는 코드를 실행하게 될 경우 일부 오류나 무한루프 등에 빠질 수 있는 위험이 있습니다.

### Launch 패키지 실습

#### 1 - 1. 노드파일 확인 (talker_node.py) 

(py_launch_example/py_launch_example/talker_node.py 참고.)

In [ ]:
import rclpy
from rclpy.node import Node
from std_msgs.msg import String

class TalkerNode(Node):
    def __init__(self):
        super().__init__('talker_node')
        self.publisher = self.create_publisher(String, 'chatter', 10)
        self.timer = self.create_timer(1.0, self.timer_callback)
        self.count = 0

    def timer_callback(self):
        msg = String()
        msg.data = f'Hello ROS2 Launch {self.count}'
        self.publisher.publish(msg)
        self.get_logger().info(f'Publish: {msg.data}')
        self.count += 1

def main(args=None):
    rclpy.init(args=args)
    node = TalkerNode()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        pass
    finally:
        node.destroy_node()
        rclpy.shutdown()

if __name__ == '__main__':
    main()

#### 1 - 2. 노드파일 확인 (listener_node.py) 

(py_launch_example/py_launch_example/listener_node.py 참고.)

In [ ]:
import rclpy
from rclpy.node import Node
from std_msgs.msg import String

class ListenerNode(Node):
    def __init__(self):
        super().__init__('listener_node')
        self.subscription = self.create_subscription(
            String,
            'chatter',
            self.listener_callback,
            10
        )
    def listener_callback(self, msg):
        self.get_logger().info(f'Receive: {msg.data}')

def main(args=None):
    rclpy.init(args=args)
    node = ListenerNode()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        pass
    finally:
        node.destroy_node()
        rclpy.shutdown()

if __name__ == '__main__':
    main()

#### 1 - 3. 빌드 및 실행

setup.py 내용 추가하고 빌드 작업.

In [ ]:
entry_points={
    'console_scripts': [
        'talker_node = py_launch_example.talker_node:main',
        'listener_node = py_launch_example.listener_node:main',
    ],
}

터미널 2개 띄운 후 양쪽 모두 

```bash
source install/setup.bash
```

이후 각각 아래 명령어를 한개씩 실행한다.

```bash
ros2 run py_launch_example talker_node
ros2 run py_launch_example listener_node
```

송수신이 잘 되는지 확인한다.

#### 2. launch파일 확인 (talker_listener.launch.py) 

(py_launch_example/launch/talker_listener.launch.py 참고.)

In [ ]:
from launch import LaunchDescription
from launch_ros.actions import Node

def generate_launch_description():
    talker_node = Node(
        package='py_launch_example',
        executable='talker_node',
        name='talker'
    )

    listener_node = Node(
        package='py_launch_example',
        executable='listener_node',
        name='listener'
    )

    return LaunchDescription([
        talker_node,
        listener_node
    ])

In [ ]:
from launch import LaunchDescription
from launch_ros.actions import Node

def generate_launch_description():
    talker_node = Node(
        package='py_launch_example',
        executable='talker_node',
        name='talker',
        namespace='robot1',
        remappings=[
            ('chatter', 'robot_chatter')
        ]
    )

    listener_node = Node(
        package='py_launch_example',
        executable='listener_node',
        name='listener',
        namespace='robot1',
        remappings=[
            ('chatter', 'robot_chatter')
        ]
    )

    return LaunchDescription([
        talker_node,
        listener_node
    ])